In [1]:
from transformers import pipeline 
import pandas as pd 

In [2]:
filepath = '../Project_btc_sentiment/ compile_data/data_2016-12-06_2020-12-22.csv' 

In [3]:
df = pd.read_csv(filepath)

In [4]:
df.shape

(7947, 40)

In [5]:
df_sel = df[['date','TITLE','SUBTITLE','PUBLISHED_ON']]

In [6]:
# concatenate title and subtitle to together
df_sel['title_subtitle'] = df_sel[['TITLE','SUBTITLE']].fillna('').agg(','.join, axis = 1).str.strip()

C:\Users\user\AppData\Local\Temp\ipykernel_12024\573294513.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_sel['title_subtitle'] = df_sel[['TITLE','SUBTITLE']].fillna('').agg(','.join, axis = 1).str.strip()


In [7]:
# Load Finbert transformer
from transformers import pipeline
pipe = pipeline("text-classification", model="ProsusAI/finbert")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [10]:
df_sel.head(2)

,date,TITLE,SUBTITLE,PUBLISHED_ON,title_subtitle
0,2016-12-31,Not Just Bitcoin: The Top 7 Cryptocurrencies A...,CoinDesk contributor Frederick Reese gives an ...,1483198607,Not Just Bitcoin: The Top 7 Cryptocurrencies A...
1,2016-12-31,2016: The Year of Blockchain Hubris,"2016 may have been a big year for blockchain, ...",1483160183,"2016: The Year of Blockchain Hubris,2016 may h..."


In [11]:
texts = df_sel['title_subtitle'].tolist()

resuts = pipe(texts, batch_size = 64)

In [12]:
df_sel['finbert_label'] = [r['label'] for r in resuts ]
df_sel['finbert_score'] = [r['score'] for r in resuts ]

C:\Users\user\AppData\Local\Temp\ipykernel_12024\2413491440.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_sel['finbert_label'] = [r['label'] for r in resuts ]
C:\Users\user\AppData\Local\Temp\ipykernel_12024\2413491440.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_sel['finbert_score'] = [r['score'] for r in resuts ]


In [13]:
df_sel.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7947 entries, 0 to 7946
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   date            7947 non-null   object 
 1   TITLE           7947 non-null   object 
 2   SUBTITLE        7866 non-null   object 
 3   PUBLISHED_ON    7947 non-null   int64  
 4   title_subtitle  7947 non-null   object 
 5   finbert_label   7947 non-null   object 
 6   finbert_score   7947 non-null   float64
dtypes: float64(1), int64(1), object(5)
memory usage: 434.7+ KB


In [14]:
pd.set_option('display.max_colwidth', None)

In [15]:
# define function to compute sentiment on -1 to 1 scale 
# positive sentiment, will yield 0-1 score 
# neutral, will yield 0 
def std_score(row): 

    label = row.get('finbert_label')
    score = row.get('finbert_score')
    
    if pd.isna(label) or pd.isna(score):  
        return np.nan 
        
    label = label.lower()
    score = float(score) 

    if label == 'positive' :
        return score 
    elif label == 'negative' :
        return -score 
    else : # neutral
        return 0.0 


df_sel['sentiment_score'] = df_sel[['finbert_label','finbert_score']].apply(std_score, axis = 1)
    

C:\Users\user\AppData\Local\Temp\ipykernel_12024\2736805221.py:23: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_sel['sentiment_score'] = df_sel[['finbert_label','finbert_score']].apply(std_score, axis = 1)


In [19]:
df_sel.tail(3)

,date,TITLE,SUBTITLE,PUBLISHED_ON,title_subtitle,finbert_label,finbert_score,sentiment_score
7944,2020-12-20,Dogecoin Jumps 20% After Musk's Twitter Shout-Out; Bitcoin Joke Spurs Dialogue With Saylor,"Musk fired off a series of bitcoin-related tweets, too.",1608463146,"Dogecoin Jumps 20% After Musk's Twitter Shout-Out; Bitcoin Joke Spurs Dialogue With Saylor,Musk fired off a series of bitcoin-related tweets, too.",negative,0.739769,-0.739769
7945,2020-12-19,Tiny Capital's Wilkinson Shows Interest in Bitcoin,Wilkinson's tweeted inquiry started a lively debate on the merits of the most valuable cryptocurrency.,1608410959,"Tiny Capital's Wilkinson Shows Interest in Bitcoin,Wilkinson's tweeted inquiry started a lively debate on the merits of the most valuable cryptocurrency.",neutral,0.609603,0.000000
7946,2020-12-19,"Bitcoin Tops $24K, Setting New All-Time High","Bitcoin cut through $24,000 Saturday afternoon, setting a new record high as the leading cryptocurrency's ongoing rally continues.",1608395560,"Bitcoin Tops $24K, Setting New All-Time High,Bitcoin cut through $24,000 Saturday afternoon, setting a new record high as the leading cryptocurrency's ongoing rally continues.",positive,0.897128,0.897128


In [20]:
min_max_date = df_sel['date'].agg(['min', 'max'])
min_max_date

min    2016-12-06
max    2020-12-22
Name: date, dtype: object

In [21]:
print(f"transformed_{min_max_date['min']}_{min_max_date['max']}.csv")

transformed_2016-12-06_2020-12-22.csv


In [22]:
df_sel.to_csv(f"transformed_{min_max_date['min']}_{min_max_date['max']}.csv")
print(f'Save successfully')

Save successfully
